# Size the checkout test

*Data Analysis in Python · Topic 9.5*

In the comparable users, you found B's checkout +3.6% on A's, but you couldn't tell that lift from chance. Your product manager asks how long the test would have needed to run to detect a lift that size.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that reads the test's two files, the cell that builds the `users` table (topic 9.1.'s steps) and the cell that builds the `fixed_users` table: the users outside the in-app browsers (topic 9.3.'s steps).
2. In the first cell marked ✏️, create, from the `fixed_users` table:
   - `baseline_rate`: A's conversion rate
   - `observed_lift`: B's conversion rate lift over A, as a fraction
3. In the second cell marked ✏️, create `users_needed`: the users per variant a test needs to detect `observed_lift` from `baseline_rate`, at a significance level of 0.05 and a power of 0.8, rounded up to a whole user.
4. In the last cell marked ✏️, create:
   - `users_per_week`: the `fixed_users` table's users per variant per week, over the test's `TEST_WEEKS` weeks
   - `weeks_needed`: the whole weeks the test would need to reach `users_needed` in each variant, rounded up
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `baseline_rate` is about `0.0489`, and `observed_lift` about `0.0358`. `users_needed` is `242212`. `users_per_week` is `6350.75`, and `weeks_needed` is `39`.

## Parameters

In [ ]:
import math

import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
IN_APP_BROWSERS = ["Android Webview", "Safari (in-app)"]
TEST_WEEKS = 6

In [ ]:
# The test's two files, with the user IDs as text. Run it, and leave it as it is.
assignments = pd.read_csv(DATA_URL + "ab_test_assignments.csv", dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(DATA_URL + "ab_test_conversions.csv", dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
assignments.shape, conversions.shape

In [ ]:
# One row per user in the test, with converted and revenue (topic 9.1). Run it, and leave it as it is.
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
users.shape

In [ ]:
# The users outside the in-app browsers, which B couldn't log (topic 9.3). Run it, and leave it as it is.
fixed_users = users[~users["browser"].isin(IN_APP_BROWSERS)]
fixed_users.shape

## The starting point

In [ ]:
rates = fixed_users.groupby("variant")["converted"].mean()
baseline_rate = rates["A"]
observed_lift = rates["B"] / rates["A"] - 1
baseline_rate, observed_lift

## Users per variant

In [ ]:
effect = proportion_effectsize(baseline_rate * (1 + observed_lift), baseline_rate)
users_needed = math.ceil(NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8))
users_needed

## Weeks

In [ ]:
users_per_week = len(fixed_users) / 2 / TEST_WEEKS
weeks_needed = math.ceil(users_needed / users_per_week)
users_per_week, weeks_needed

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.000001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    baseline, lift = found.get("baseline_rate"), found.get("observed_lift")
    if not number(baseline) or not number(lift):
        results.append((False, "baseline_rate and observed_lift should be numbers: A's conversion rate in the fixed_users table, and B's rate divided by A's, minus 1."))
    elif near(baseline, 0.045725):
        results.append((False, "baseline_rate is 0.0457: A's rate among all users, in-app browsers included. Use the fixed_users table, the users the test could compare."))
    elif near(baseline, 0.050694):
        results.append((False, "baseline_rate is 0.0507: that's B's rate. The baseline is the control's, A's."))
    elif not near(baseline, 0.048942):
        results.append((False, f"baseline_rate should be A's conversion rate in the fixed_users table, about 0.0489. It is {baseline:.4f}."))
    elif near(lift, 0.108668):
        results.append((False, "observed_lift is 0.1087: the lift of all users. Use the fixed_users table."))
    elif not near(lift, 0.035786):
        results.append((False, f"observed_lift should be B's conversion rate divided by A's, minus 1, in the fixed_users table: about 0.0358. It is {lift:.4f}."))
    else:
        results.append((True, f"A's comparable users converted at {baseline:.2%}, and B's {lift:+.1%} on that."))

    needed = found.get("users_needed")
    if not number(needed):
        results.append((False, f"users_needed should be a number: math.ceil(NormalIndPower().solve_power(effect_size=..., alpha=0.05, power=0.8)). It is {needed!r}."))
    elif near(needed, 242211.017, tolerance=0.01):
        results.append((False, "users_needed is 242,211.02: round it up to a whole user with math.ceil(), 242,212. Rounding down would leave the test just short of its power."))
    elif needed in (484423, 484424):
        results.append((False, f"users_needed is {needed:,}: both variants together. The brief asks for users per variant: solve_power() gives that directly."))
    elif needed in (190790, 190789):
        results.append((False, "users_needed is 190,790: that's for a one-sided test. Topic 9.4.'s test is two-sided, so plan for that one."))
    elif needed in (324253, 324252):
        results.append((False, "users_needed is 324,253: that's for a power of 0.9. The brief asks for 0.8."))
    elif needed in (29156, 29155):
        results.append((False, "users_needed is 29,156: the sample size for all users' +10.9% lift. Use the fixed_users table's baseline and lift."))
    elif not (needed == 242212 or needed in (242226, 242227)):
        results.append((False, f"users_needed should be the users per variant to detect a {lift:+.1%} lift from {baseline:.2%}, at alpha 0.05 and power 0.8, rounded up: 242,212. It is {needed!r}."))
    else:
        results.append((True, f"Detecting a +3.6% lift from 4.89% takes {needed:,} users per variant: {needed / 38104.5:.1f} times what each variant had."))

    per_week, weeks = found.get("users_per_week"), found.get("weeks_needed")
    if not number(per_week):
        results.append((False, f"users_per_week should be a number: len(fixed_users) / 2 / TEST_WEEKS. It is {per_week!r}."))
    elif near(per_week, 12701.5, tolerance=0.01):
        results.append((False, "users_per_week is 12,701.5: both variants together. Divide by 2 as well, because users_needed is per variant."))
    elif near(per_week, 6671.1667, tolerance=0.001):
        results.append((False, "users_per_week is 6,671.17: that's all users. The in-app browsers' users are left out of the comparison, so they don't count toward it."))
    elif not near(per_week, 6350.75, tolerance=0.001):
        results.append((False, f"users_per_week should be the fixed_users table's users, divided by 2 variants and {found.get('TEST_WEEKS', 6)} weeks: 6,350.75. It is {per_week!r}."))
    elif not number(weeks):
        results.append((False, f"weeks_needed should be a number: math.ceil(users_needed / users_per_week). It is {weeks!r}."))
    elif near(weeks, 38.1395, tolerance=0.001):
        results.append((False, "weeks_needed is 38.14: round it up to whole weeks with math.ceil(). A test stopped after 38 weeks is short of its sample size."))
    elif weeks != 39:
        results.append((False, f"weeks_needed should be users_needed divided by users_per_week, rounded up: 39. It is {weeks!r}."))
    else:
        results.append((True, f"At {per_week:,.0f} users per variant a week, the test needed {weeks} weeks, not {found.get('TEST_WEEKS', 6)}. It could never have detected a lift this small."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
rates = fixed_users.groupby("variant")["converted"].mean()
baseline_rate = rates["A"]
observed_lift = rates["B"] / rates["A"] - 1
baseline_rate, observed_lift
```

```python
effect = proportion_effectsize(baseline_rate * (1 + observed_lift), baseline_rate)
users_needed = math.ceil(NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8))
users_needed
```

```python
users_per_week = len(fixed_users) / 2 / TEST_WEEKS
weeks_needed = math.ceil(users_needed / users_per_week)
users_per_week, weeks_needed
```

</details>